# 📝 과제 LV1(기초): 타이타닉 탑승객 대시보드

Streamlit 위젯·레이아웃·차트를 하나씩 익힙니다. `app.py` 의 같은 번호 자리에 코드를 채우세요.

| | |
| --- | --- |
| 데이터 | `data/titanic.csv` (탑승객 891명) |
| 실행 | 루트에서 `uv run streamlit run 과제_LV1_기초/app.py` |
| 주요 열 | `survived`(0/1) · `class`(First/Second/Third) · `sex`(male/female) · `age` · `fare` |

`df` 는 제공된 `load_data()` 로 이미 불러와 있습니다.

**전체 규칙**: 상단 지표(문제 3·4)는 **언제나 전체 891명 기준**입니다. 필터(문제 5~7)는 표·그래프(문제 8·9)에만 적용합니다. 그래서 코드도 지표를 먼저 그리고 그 뒤에 필터를 만듭니다.


### 완성 화면

![LV1 완성 대시보드](../images/lv1/lv1_overview.png)


## 1. 페이지 설정과 제목

- `st.set_page_config(page_title="타이타닉 대시보드", page_icon="🚢", layout="wide")` 를 **st 명령 중 가장 먼저** 호출
- `st.title` 로 제목, `st.caption` 으로 한 줄 설명

**확인**: 브라우저 탭 제목이 "타이타닉 대시보드" 이고 화면 상단에 제목과 회색 설명이 보인다.


## 2. 데이터 살펴보기

- `st.write` 로 전체 행·열 개수를 한 문장으로 표시 (예: `전체 891명, 11개 열`)
- `st.dataframe(df.head(10))` 으로 앞부분만 표로 표시

**확인**: 문장 아래에 10행짜리 표가 나온다.


## 3. 핵심 지표 하나 (`st.metric`)

- `st.metric` 으로 "탑승객 수" 지표 표시. 값은 `len(df)` 기준

**확인**: `탑승객 수 891명` 형태의 큰 숫자가 보인다.


## 4. 지표 여러 개를 나란히 (`st.columns`)

- `st.columns(3)` 으로 3칸을 만들고 칸마다 `st.metric` 배치
- 지표 3개: 생존자 수(`df["survived"].sum()`) · 생존율(`df["survived"].mean() * 100`, 소수 1자리 + `%`) · 평균 요금(`df["fare"].mean()`, 앞에 `$`)
- 계산 대상은 `filtered` 가 아니라 **`df`** (지표는 전체 기준이고, 아직 필터를 만들기 전입니다)

**확인**: `생존자 수 342명 | 생존율 38.4% | 평균 요금 $32.2` 가 한 줄에 나란히 보인다.


## 5. 객실 등급 필터 (`st.multiselect`)

- 사이드바(`st.sidebar` 또는 `with st.sidebar:`)에 `st.multiselect` 로 객실 등급 선택. 선택지 `["First", "Second", "Third"]`, 기본값은 전체 선택
- 선택한 등급만 남긴 결과를 **`filtered`** 변수에 담기 (문제 6~8이 이 변수를 이어서 씁니다)

**확인**: "Third" 만 남기면 이후 표·그래프가 3등급 승객만 보여 준다. 상단 지표는 전체 기준이라 그대로다.


## 6. 성별 필터 (`st.radio`)

- 사이드바에 `st.radio` 로 성별 선택: `["전체", "male", "female"]`, `horizontal=True`
- "전체" 가 아닐 때만 해당 성별로 **`filtered` 를 추가로** 거르기 (`df` 로 되돌아가면 앞 필터가 풀립니다)

**확인**: "female" 을 고르면 여성 승객만 남고, 등급 필터도 함께 유지된다.


## 7. 나이 범위 필터 (`st.slider`)

- 사이드바에 `st.slider` 로 나이 범위 선택. 범위 0~100, 기본값 `(0, 80)` (튜플을 주면 범위 슬라이더가 됩니다)
- 선택 범위 안의 나이만 남기도록 `filtered` 를 추가로 거르기 (`Series.between` 활용)

**확인**: 20~40 으로 좁히면 그 나이대만 남는다.


## 8. 표와 차트를 탭으로 분리 (`st.tabs`)

- `st.tabs(["데이터 표", "시각화"])` 로 탭 2개 생성
- "데이터 표" 탭: `st.dataframe(filtered)`
- "시각화" 탭: 문제 9의 그래프

**확인**: 탭을 눌러 표 화면과 그래프 화면을 전환할 수 있다.


## 9. 객실 등급별 생존자 그래프 (`st.pyplot` + seaborn)

- seaborn `countplot` 으로 x축 `class`, 색 구분 `hue="survived"`. 데이터는 `filtered`
- `fig, ax = plt.subplots()` 로 그림을 만들고 `ax=ax` 로 그린 뒤 `st.pyplot(fig)`, 마지막에 `plt.close(fig)`
- 축 라벨을 한글로: x축 "객실 등급", y축 "인원수"

**확인**:

![목표 화면: 시각화 탭](../images/lv1/lv1_chart.png)


## 10. 조회수 카운터 (`st.session_state`)

- `st.session_state` 의 `views` 를 0으로 초기화. **키가 없을 때만** (`if "views" not in st.session_state:`)
- `st.button("조회수 올리기")` 를 누르면 `views` 를 1 증가
- `st.write` 로 현재 조회수 표시

**확인**: 버튼을 누를 때마다 숫자가 1씩 올라가고, 다른 위젯을 조작해도 값이 유지된다.
